# 🎮 Crafter x DreamerV3 训练与评测指南 (Google Colab)

本 Notebook 专为在 **Google Colab (免费 T4 / V100 / A100 GPU)** 上训练 **DreamerV3** 强化学习世界模型而设计：
- 🌟 **支持全新起步训练** (如 100k 步基准验证)
- 🔄 **支持一键断点续训 (Resume Training)**：自动从 Google Drive 挂载并恢复已有进度 (如 `crafter_dreamerv3_100k`)，无缝继续训练到 200k / 300k+ 步！

### 🛠️ 整体流程概览:
1. **检查 GPU 算力资源** (确认 T4/A100 与 CUDA 驱动正常)
2. **挂载 Google Drive 与自动恢复检查点** (智能检测已有的 100k 步存档并恢复至本地高速 SSD)
3. **一键拉取仓库与核心依赖** (danijar/dreamerv3 + crafter + JAX 依赖 + JAX jit 兼容补丁)
4. **优化 JAX 显存策略** (按需申请显存，防止 OOM)
5. **启动 / 继续 DreamerV3 训练任务** (自由设定目标步数如 200k，实时流式输出日志，训练后自动备份至云盘)
6. **训练成果评估与 22 项成就解锁可视化** (动态绘制最新步数得分曲线与成就解锁柱状图)
7. **打包下载最新权重与评测日志**


## 1. 检查 GPU 算力环境
Colab 菜单 -> **代码执行程序 (Runtime)** -> **更改运行时类型 (Change runtime type)** -> 硬件加速器选择 **T4 GPU**。

In [1]:
import shutil
import subprocess

# 1. 检查并打印 GPU 硬件状态 (显式捕获输出以兼容 VS Code 和 Colab 网页端)
if shutil.which("nvidia-smi"):
    res = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=False)
    if res.stdout:
        print(res.stdout, flush=True)
    if res.stderr:
        print(res.stderr, flush=True)
else:
    print("未检测到 nvidia-smi 工具（当前实例未分配 GPU 硬件或处于纯 CPU 运行时）。", flush=True)

# 2. 检查 JAX 加速设备
print("\n--- JAX 计算设备检测 ---", flush=True)
try:
    import jax
    devices = jax.devices()
    print("当前 JAX 可用计算设备:", devices, flush=True)
    has_gpu = any(d.platform == "gpu" for d in devices)
except ImportError:
    has_gpu = False
    print("当前环境尚未安装 JAX，将在下一步依赖安装中自动配置。", flush=True)

if not has_gpu:
    print("\n" + "!" * 70, flush=True)
    print("【重要提示】当前未检测到 GPU 算力加速器（处于 CPU 模式）！", flush=True)
    print("若在 Google Colab 上运行，请切换为 GPU 运行时：", flush=True)
    print("👉 顶部菜单: 代码执行程序 (Runtime) -> 更改运行时类型 (Change runtime type)", flush=True)
    print("👉 硬件加速器选择: T4 GPU -> 点击 保存 (Save)", flush=True)
    print("!" * 70 + "\n", flush=True)
else:
    print("\n✅ GPU 加速器已就绪，可顺畅继续执行后续训练步骤！", flush=True)


Tue Oct 27 12:40:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. 挂载 Google Drive 并自动恢复历史检查点 (支持断点续训)
> 💡 **断点续训核心机制**：
> - 自动挂载 Google Drive，并检测 `我的云端硬盘/crafter_dreamerv3_100k` 是否存在历史检查点 (`checkpoint.ckpt` / `metrics.jsonl` 等)。
> - 若存在，则自动将历史权重和经验回放数据复制到 Colab 本地高速 SSD (`/content/output/crafter_dreamerv3_100k`)，并读取已完成的训练步数。
> - 若不存在或为初次训练，则自动新建目录准备从 0 步开始全新训练。


In [2]:
import os
import json
import shutil
from pathlib import Path
from google.colab import drive

# 1. 挂载 Google 云端硬盘
print("正在请求挂载 Google Drive...")
drive.mount('/content/drive')

gdrive_path = Path("/content/drive/MyDrive/crafter_dreamerv3_100k")
local_path = Path("/content/output/crafter_dreamerv3_100k")
local_path.mkdir(parents=True, exist_ok=True)

# 2. 智能检测 Google Drive 上的历史存档
resume_ready = False
current_step = 0

if gdrive_path.exists():
    print(f"\n[检测到云盘目录] {gdrive_path}")
    
    # 检查是否有 checkpoint 或 metrics 文件
    ckpt_files = list(gdrive_path.glob("*.ckpt"))
    has_metrics = (gdrive_path / "metrics.jsonl").exists()
    
    if ckpt_files or has_metrics:
        print("[恢复检查点] 正在从 Google Drive 恢复历史训练权重与经验回放数据至本地高速 SSD...")
        shutil.copytree(gdrive_path, local_path, dirs_exist_ok=True)
        print("[恢复完成] 历史数据已完整同步至本地运行环境。")
        
        metrics_file = local_path / "metrics.jsonl"
        if metrics_file.exists():
            records = []
            with open(metrics_file, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if line:
                        try:
                            records.append(json.loads(line))
                        except Exception:
                            pass
            if records:
                current_step = int(records[-1].get("step", 0))
                print(f"📊 当前已完成历史训练步数: {current_step:,} steps")
                resume_ready = True
    else:
        print("[提示] 云盘目录存在但未发现历史检查点文件，将作为全新训练开始。")
else:
    print(f"\n[全新训练] 未在 Google Drive 找到历史存档 ({gdrive_path})，本次将从 0 步开始全新训练。")

print("\n" + "=" * 60)
if resume_ready:
    print(f"🚀 已就绪【断点续训】模式！历史进度: {current_step:,} steps")
    print(f"👉 下方训练单元格将自动从第 {current_step:,} 步向后继续训练！")
    print(f"👉 推荐将目标步数设置为 200,000 (即增量再训 100,000 步)")
else:
    print("🌱 已就绪【全新训练】模式！将从 0 步开始训练至目标步数 (默认 100,000 步)")
print("=" * 60)


Mounted at /content/drive

[检测到云盘目录] /content/drive/MyDrive/crafter_dreamerv3_100k
[恢复检查点] 正在从 Google Drive 恢复历史训练权重与经验回放数据至本地高速 SSD...
[恢复完成] 历史数据已完整同步至本地运行环境。
📊 当前已完成历史训练步数: 47,110 steps

🚀 已就绪【断点续训】模式！历史进度: 47,110 steps
👉 下方训练单元格将自动从第 47,110 步向后继续训练！
👉 推荐将目标步数设置为 200,000 (即增量再训 100,000 步)


## 3. 拉取 DreamerV3 官方源码与安装依赖
本步骤自动克隆官方 `danijar/dreamerv3` 仓库，安装 `crafter` 与核心依赖，并自动应用新版 JAX 关键字参数兼容补丁。

In [3]:
import os
import subprocess
import sys
from pathlib import Path

# 1. 克隆官方 DreamerV3 仓库
repo_dir = Path("/content/dreamerv3")
if not (repo_dir / "dreamerv3" / "main.py").exists():
    subprocess.run(["git", "clone", "https://github.com/danijar/dreamerv3.git", str(repo_dir)], check=True)
else:
    print("DreamerV3 仓库已就绪。")

# 2. 安装核心依赖包
pkgs = [
    "crafter",
    "elements",
    "portal",
    "ninjax",
    "optax",
    "chex",
    "einops",
    "ruamel.yaml",
    "scope",
    "matplotlib",
    "pandas",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

# 3. 自动注入新版 JAX jit 兼容补丁
main_file = repo_dir / "dreamerv3" / "main.py"
if main_file.exists():
    content = main_file.read_text(encoding="utf-8")
    patch_marker = "# === JAX COMPATIBILITY MONKEY PATCH ==="
    if patch_marker not in content:
        patch_lines = [
            patch_marker,
            "import jax",
            "_orig_jit = jax.jit",
            "def _compat_jit(fun=None, *args, **kwargs):",
            "    if not args:",
            "        return _orig_jit(fun, **kwargs)",
            "    names = ['in_shardings', 'out_shardings', 'static_argnums', 'static_argnames', 'donate_argnums']",
            "    for val, name in zip(args, names):",
            "        if val is not None and name not in kwargs and name != 'static_argnames':",
            "            kwargs[name] = val",
            "    return _orig_jit(fun, **kwargs)",
            "jax.jit = _compat_jit",
            "",
        ]
        patch_code = "\n".join(patch_lines)
        main_file.write_text(patch_code + "\n" + content, encoding="utf-8")
        print("已成功注入 JAX jit 兼容补丁！")

print("\n所有核心依赖与环境补丁安装完成！")


已成功注入 JAX jit 兼容补丁！

所有核心依赖与环境补丁安装完成！


## 4. JAX 显存策略与环境变量配置
JAX 默认会直接预分配 90% 显存。在单 GPU 且需多任务协作的 Colab 上，关闭预分配以按需申请显存，提升稳定性。

In [4]:
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.85"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
print("JAX 显存优化参数已注入系统环境。")


JAX 显存优化参数已注入系统环境。


## 5. 启动 / 继续 DreamerV3 训练

### 📌 关键配置与继续训练说明:
- **`TARGET_STEPS` (目标总步数)**:
  - 若这是**首次训练**：推荐设为 `100000` (100k 步)。
  - 若这是**继续训练**：必须大于当前已完成步数！例如已完成 100k 步，继续训练推荐设为 `200000` (再练 100k 步，总计 200k 步)。
  - **DreamerV3 断点续训原理**：当 `--logdir` 指向包含已有检查点的目录时，DreamerV3 会自动无缝加载模型权重、优化器状态和经验池，直接从上次记录的 step 继续向后训练，直至达到 `--run.steps`。
- **自动云盘持久化安全机制**：
  - 训练结束或用户中断时，自动将最新权重、经验池与日志增量同步回 Google Drive (`/content/drive/MyDrive/crafter_dreamerv3_100k`)，防止掉线丢失！


In [ ]:
import os
import sys
import json
import shutil
import subprocess
from pathlib import Path

output_dir = Path("/content/output/crafter_dreamerv3_100k")
gdrive_dir = Path("/content/drive/MyDrive/crafter_dreamerv3_100k")
repo_dir = Path("/content/dreamerv3")
output_dir.mkdir(parents=True, exist_ok=True)

# ==================== 用户参数配置 ====================
# 目标总步数 (Total Environment Steps):
# 如果历史已训练 100k 步，继续训练请设置为 200000 (即训练到 200k 步)
# 可根据需要修改为 150000, 200000, 300000 等
TARGET_STEPS = 200000

# 模型规模预设 (必须保持与历史存档一致，勿随意更改以免网络权重形状不匹配)
MODEL_SIZE = "size12m"

# 样本复用比率 (默认: 512)
TRAIN_RATIO = 512
# =====================================================

# 1. 检测当前已有的步数
current_step = 0
metrics_file = output_dir / "metrics.jsonl"
if metrics_file.exists():
    try:
        with open(metrics_file, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    record = json.loads(line)
                    current_step = int(record.get("step", current_step))
    except Exception as e:
        print(f"[提示] 读取历史 metrics 时异常: {e}")

print("=" * 60)
print(f"📊 当前存档进度:     {current_step:,} steps")
print(f"🎯 设定目标总步数:   {TARGET_STEPS:,} steps")

if TARGET_STEPS <= current_step:
    print("\n" + "!" * 60)
    print(f"【提醒】设定目标步数 ({TARGET_STEPS:,}) <= 当前已完成步数 ({current_step:,})！")
    print(f"DreamerV3 将认为训练已达标而直接退出。")
    auto_new_target = current_step + 100000
    print(f"👉 自动为您将目标步数递增至: {auto_new_target:,} 步 (增训 100,000 步)")
    print("!" * 60 + "\n")
    TARGET_STEPS = auto_new_target

incremental_steps = TARGET_STEPS - current_step
print(f"🚀 本次将执行增量训练: {incremental_steps:,} steps")
print("=" * 60 + "\n")

# 2. 构造 DreamerV3 运行命令
cmd = [
    sys.executable,
    f"{repo_dir}/dreamerv3/main.py",
    "--configs", "crafter", MODEL_SIZE,
    "--task", "crafter_reward",
    "--logdir", str(output_dir),
    "--run.steps", str(TARGET_STEPS),
    "--run.train_ratio", str(TRAIN_RATIO),
    "--run.log_every", "30",
    "--run.save_every", "300",
    "--jax.prealloc", "False",
]

env = os.environ.copy()
env["PYTHONPATH"] = str(repo_dir) + os.pathsep + env.get("PYTHONPATH", "")

print(f"[执行命令] {' '.join(cmd)}\n")

# 3. 启动并流式监听训练
try:
    proc = subprocess.Popen(
        cmd,
        cwd=str(repo_dir),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    for line in iter(proc.stdout.readline, ""):
        print(line, end="", flush=True)
    proc.stdout.close()
    return_code = proc.wait()
    if return_code != 0:
        print(f"\n[警告] 训练进程退出，返回码: {return_code}")
    else:
        print(f"\n🎉 恭喜！训练成功达成目标步数 ({TARGET_STEPS:,} 步)！")
except KeyboardInterrupt:
    print("\n[中断] 接收到手动中断信号 (Ctrl+C)。正在终止训练并保护检查点...")
    if 'proc' in locals():
        proc.terminate()
        proc.wait()

# 4. 训练结束或中断时，自动将最新权重同步保存到 Google Drive
if gdrive_dir.parent.exists():
    print("\n[云盘备份] 正在将最新权重、经验池与评估日志同步至 Google Drive...")
    try:
        shutil.copytree(output_dir, gdrive_dir, dirs_exist_ok=True)
        print(f"✅ Google Drive 同步成功: {gdrive_dir}")
    except Exception as e:
        print(f"[备份警告] 同步至云盘失败: {e}")


📊 当前存档进度:     47,110 steps
🎯 设定目标总步数:   200,000 steps
🚀 本次将执行增量训练: 152,890 steps

[执行命令] /usr/bin/python3 /content/dreamerv3/dreamerv3/main.py --configs crafter size12m --task crafter_reward --logdir /content/output/crafter_dreamerv3_100k --run.steps 200000 --run.train_ratio 512 --run.log_every 30 --run.save_every 300 --jax.prealloc False

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/content/dreamerv3/embodied/jax/agent.py:112: DeprecationWarning: `with mesh:` context manager has been deprecated. Please use `with jax.set_mesh(mesh):` instead.
  with self.train_mesh:
/usr/local/lib/python3.13/dist-packages/optax/_src/linear_algebra.py:44: DeprecationWarning: optax.global_no

## 6. 训练结果可视化与 Crafter 成就解锁分析
解析 `metrics.jsonl`，绘制训练得分曲线与 22 项生存成就达成率（自适应最新训练步数）。


In [ ]:
import json
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

metrics_file = Path("/content/output/crafter_dreamerv3_100k/metrics.jsonl")
if not metrics_file.exists():
    files = list(Path("/content/output/crafter_dreamerv3_100k").glob("**/metrics.jsonl"))
    if files:
        metrics_file = files[0]

if metrics_file.exists():
    records = []
    with open(metrics_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    records.append(json.loads(line))
                except Exception:
                    pass

    df = pd.DataFrame(records)
    print(f"成功载入 {len(df)} 条指标记录！")

    last_row = df.iloc[-1]
    latest_step = int(last_row.get("step", 0))
    latest_score = float(last_row.get("episode/score", last_row.get("score", 0.0)))
    latest_len = float(last_row.get("episode/length", last_row.get("length", 0.0)))

    print(f"📊 最新总训练步数: {latest_step:,} steps")
    print(f"🏆 当前最新平均分: {latest_score:.2f}")
    print(f"⏱️ 当前平均存活长: {latest_len:.1f} steps")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    score_col = "episode/score" if "episode/score" in df.columns else "score"
    len_col = "episode/length" if "episode/length" in df.columns else "length"

    if score_col in df.columns and "step" in df.columns:
        axes[0].plot(df["step"], df[score_col], color="#2563EB", linewidth=2, label="Score")
        axes[0].set_title(f"Crafter Score vs Steps (Up to {latest_step:,})", fontsize=13, fontweight="bold")
        axes[0].set_xlabel("Steps")
        axes[0].set_ylabel("Score")
        axes[0].grid(True, linestyle="--", alpha=0.5)
        axes[0].legend()

    if len_col in df.columns and "step" in df.columns:
        axes[1].plot(df["step"], df[len_col], color="#10B981", linewidth=2, label="Episode Length")
        axes[1].set_title(f"Episode Survival Length (Up to {latest_step:,})", fontsize=13, fontweight="bold")
        axes[1].set_xlabel("Steps")
        axes[1].set_ylabel("Survival Length (Steps)")
        axes[1].grid(True, linestyle="--", alpha=0.5)
        axes[1].legend()

    plt.tight_layout()
    plt.show()

    # 提取 Crafter 成就达成情况
    achievements = {}
    for col in df.columns:
        if "achievement" in col.lower():
            ach_name = col.split("/")[-1].replace("achievement_", "")
            achievements[ach_name] = float(last_row[col])

    if achievements:
        ach_series = pd.Series(achievements).sort_values(ascending=True)
        plt.figure(figsize=(10, 8))
        colors = ["#10B981" if v > 0.05 else "#9CA3AF" for v in ach_series.values]
        bars = plt.barh(ach_series.index, ach_series.values * 100, color=colors)
        plt.title(f"Crafter Achievement Unlock Rate at {latest_step:,} Steps (%)", fontsize=14, fontweight="bold")
        plt.xlabel("Unlock Probability (%)")
        plt.grid(axis="x", linestyle="--", alpha=0.6)
        for bar in bars:
            w = bar.get_width()
            if w > 1:
                plt.text(w + 1, bar.get_y() + bar.get_height() / 2, f"{w:.1f}%", va="center", fontsize=9)
        plt.tight_layout()
        plt.show()

        unlocked = sum(1 for v in achievements.values() if v > 0.05)
        print(f"\n🎉 累计解锁成就项: {unlocked} / {len(achievements)}")
else:
    print("未找到 metrics.jsonl 文件，请先执行训练。")
